# Stock Momentum Monitor

Notebook replacement for the old Dash momentum dashboard.

- Loads data from `stock_data.db`
- Computes **RSI**, **MACD**, **Bollinger Bands**, and **pivot** metrics
- Shows ranked tables and plots a selected ticker

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

DB_PATH = Path('../stock_data.db') if Path('../stock_data.db').exists() else Path('stock_data.db')


In [ ]:
def load_prices(db_path: Path) -> pd.DataFrame:
    query = '''
        SELECT
            p.ticker,
            p.date,
            p.open,
            p.high,
            p.low,
            p.close,
            COALESCE(p.adj_close, p.close) AS price,
            p.volume,
            i.short_name,
            i.long_name,
            i.exchange,
            i.currency
        FROM ticker_prices AS p
        LEFT JOIN ticker_info AS i
            ON i.ticker = p.ticker
        ORDER BY p.ticker, p.date
    '''
    with sqlite3.connect(db_path) as conn:
        frame = pd.read_sql_query(query, conn)

    frame['date'] = pd.to_datetime(frame['date'])
    numeric_cols = ['open', 'high', 'low', 'close', 'price', 'volume']
    frame[numeric_cols] = frame[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return frame


def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)
    avg_gain = gains.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = losses.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, pd.NA)
    rsi = 100 - (100 / (1 + rs))

    flat_mask = (avg_gain == 0) & (avg_loss == 0)
    zero_loss_mask = avg_loss == 0
    rsi = rsi.mask(flat_mask, 50)
    rsi = rsi.mask(zero_loss_mask & ~flat_mask, 100)
    return rsi


def compute_indicators(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    latest_rows = []
    history_map = {}

    for ticker, ticker_frame in frame.groupby('ticker', sort=True):
        stock = ticker_frame.sort_values('date').copy()
        price = stock['price']

        stock['ema_12'] = price.ewm(span=12, adjust=False).mean()
        stock['ema_20'] = price.ewm(span=20, adjust=False).mean()
        stock['ema_26'] = price.ewm(span=26, adjust=False).mean()
        stock['ema_50'] = price.ewm(span=50, adjust=False).mean()
        stock['ema_100'] = price.ewm(span=100, adjust=False).mean()

        stock['macd_line'] = stock['ema_12'] - stock['ema_26']
        stock['macd_signal'] = stock['macd_line'].ewm(span=9, adjust=False).mean()
        stock['macd_hist'] = stock['macd_line'] - stock['macd_signal']

        stock['rsi_14'] = compute_rsi(price, 14)
        stock['rsi_7'] = compute_rsi(price, 7)
        stock['rsi_21'] = compute_rsi(price, 21)

        stock['bb_mid'] = price.rolling(20).mean()
        bb_std = price.rolling(20).std()
        stock['bb_upper'] = stock['bb_mid'] + (2 * bb_std)
        stock['bb_lower'] = stock['bb_mid'] - (2 * bb_std)
        stock['bb_width'] = stock['bb_upper'] - stock['bb_lower']
        stock['bb_pct_b'] = (price - stock['bb_lower']) / stock['bb_width'].replace(0, pd.NA)

        stock['daily_return'] = price.pct_change()

        latest = stock.iloc[-1]
        previous = stock.iloc[-2] if len(stock) > 1 else latest

        pivot = (previous['high'] + previous['low'] + previous['close']) / 3
        r1 = (2 * pivot) - previous['low']
        s1 = (2 * pivot) - previous['high']
        r2 = pivot + (previous['high'] - previous['low'])
        s2 = pivot - (previous['high'] - previous['low'])

        pivot_bias = 'above pivot' if latest['price'] >= pivot else 'below pivot'
        if pd.notna(s1) and latest['price'] <= s1:
            pivot_zone = 'at support'
        elif pd.notna(r1) and latest['price'] >= r1:
            pivot_zone = 'at resistance'
        else:
            pivot_zone = 'inside range'

        macd_state = 'bullish' if latest['macd_line'] > latest['macd_signal'] else 'bearish'
        if latest['macd_hist'] > previous['macd_hist'] and macd_state == 'bullish':
            macd_state = 'improving bullish'
        elif latest['macd_hist'] < previous['macd_hist'] and macd_state == 'bearish':
            macd_state = 'weakening bearish'

        rsi_state = 'oversold' if latest['rsi_14'] < 30 else 'overbought' if latest['rsi_14'] > 70 else 'neutral'
        bollinger_state = (
            'below lower band'
            if latest['price'] <= latest['bb_lower']
            else 'above upper band'
            if latest['price'] >= latest['bb_upper']
            else 'inside bands'
        )

        momentum_score = (
            int(latest['macd_line'] > latest['macd_signal'])
            + int(latest['macd_hist'] > 0)
            + int(latest['macd_hist'] > previous['macd_hist'])
            + int(latest['price'] > latest['ema_20'])
            + int(latest['price'] > latest['ema_50'])
            + int(50 <= latest['rsi_14'] <= 70)
            + int(latest['price'] > pivot)
        )

        latest_rows.append({
            'ticker': ticker,
            'name': latest['long_name'] or latest['short_name'] or ticker,
            'date': latest['date'].date().isoformat(),
            'price': latest['price'],
            'daily_return_pct': latest['daily_return'] * 100,
            'rsi_14': latest['rsi_14'],
            'rsi_7': latest['rsi_7'],
            'rsi_21': latest['rsi_21'],
            'rsi_state': rsi_state,
            'macd_line': latest['macd_line'],
            'macd_signal': latest['macd_signal'],
            'macd_hist': latest['macd_hist'],
            'macd_state': macd_state,
            'momentum_score': momentum_score,
            'bb_pct_b': latest['bb_pct_b'],
            'bollinger_state': bollinger_state,
            'pivot': pivot,
            'pivot_bias': pivot_bias,
            'pivot_zone': pivot_zone,
            'pivot_s1': s1,
            'pivot_s2': s2,
            'pivot_r1': r1,
            'pivot_r2': r2,
            'distance_to_pivot_pct': ((latest['price'] / pivot) - 1) * 100 if pivot else pd.NA,
            'exchange': latest['exchange'],
            'currency': latest['currency'],
        })

        history_map[ticker] = stock

    latest_df = pd.DataFrame(latest_rows).sort_values(
        ['momentum_score', 'macd_hist', 'daily_return_pct'],
        ascending=[False, False, False],
    )
    return latest_df, history_map


def format_overview(frame: pd.DataFrame):
    view = frame.copy()
    pct_cols = ['daily_return_pct', 'distance_to_pivot_pct']
    num_cols = ['price', 'rsi_14', 'bb_pct_b', 'macd_hist']
    return (
        view.style
        .format({
            'price': '{:,.2f}',
            'daily_return_pct': '{:+.2f}%',
            'rsi_14': '{:.1f}',
            'bb_pct_b': '{:.2f}',
            'macd_hist': '{:.3f}',
            'distance_to_pivot_pct': '{:+.2f}%',
        }, na_rep='-')
        .background_gradient(subset=['momentum_score'], cmap='YlGn')
        .background_gradient(subset=pct_cols, cmap='RdYlGn')
        .bar(subset=num_cols, color='#d8b36a', vmin=view[num_cols].min().min(), vmax=view[num_cols].max().max())
    )


def build_price_figure(stock: pd.DataFrame, summary: pd.Series) -> go.Figure:
    recent = stock.tail(160)
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.55, 0.2, 0.25],
    )

    fig.add_trace(go.Scatter(x=recent['date'], y=recent['price'], name='Close', line=dict(width=2.3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent['date'], y=recent['ema_20'], name='EMA 20', line=dict(width=1.4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent['date'], y=recent['ema_50'], name='EMA 50', line=dict(width=1.4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=recent['date'], y=recent['bb_upper'], name='BB Upper', line=dict(width=1, dash='dot')), row=1, col=1)
    fig.add_trace(
        go.Scatter(
            x=recent['date'],
            y=recent['bb_lower'],
            name='BB Lower',
            line=dict(width=1, dash='dot'),
            fill='tonexty',
            fillcolor='rgba(99, 179, 237, 0.10)',
        ),
        row=1,
        col=1,
    )

    for label, level in [('Pivot', summary['pivot']), ('S1', summary['pivot_s1']), ('R1', summary['pivot_r1'])]:
        if pd.notna(level):
            fig.add_hline(y=level, line=dict(dash='dash', width=1), annotation_text=label, annotation_position='top left', row=1, col=1)

    fig.add_trace(go.Scatter(x=recent['date'], y=recent['rsi_14'], name='RSI 14', line=dict(width=1.7)), row=2, col=1)
    fig.add_hline(y=70, line=dict(dash='dot', width=1), row=2, col=1)
    fig.add_hline(y=30, line=dict(dash='dot', width=1), row=2, col=1)

    colors = ['#00b386' if value >= 0 else '#ff6b6b' for value in recent['macd_hist'].fillna(0)]
    fig.add_trace(go.Bar(x=recent['date'], y=recent['macd_hist'], name='MACD Hist', marker_color=colors, opacity=0.6), row=3, col=1)
    fig.add_trace(go.Scatter(x=recent['date'], y=recent['macd_line'], name='MACD Line', line=dict(width=1.7)), row=3, col=1)
    fig.add_trace(go.Scatter(x=recent['date'], y=recent['macd_signal'], name='MACD Signal', line=dict(width=1.3, dash='dot')), row=3, col=1)

    fig.update_layout(
        title=f"{summary['ticker']} | {summary['name']}",
        template='plotly_white',
        legend=dict(orientation='h', y=1.02, x=0),
        margin=dict(l=40, r=20, t=70, b=40),
        height=860,
    )
    fig.update_xaxes(showgrid=False, zeroline=False)
    fig.update_yaxes(showgrid=True, zeroline=False)
    fig.update_yaxes(title_text='Price', row=1, col=1)
    fig.update_yaxes(title_text='RSI', range=[0, 100], row=2, col=1)
    fig.update_yaxes(title_text='MACD', row=3, col=1)
    return fig


In [ ]:
prices = load_prices(DB_PATH)
analysis, history_map = compute_indicators(prices)

print(f'Latest rows: {len(analysis)}')
print(f'Database path: {DB_PATH.resolve()}')
analysis.head()


## Full Universe

All tickers ranked by momentum score.

In [ ]:
overview_columns = [
    'ticker', 'name', 'date', 'price', 'daily_return_pct', 'rsi_14', 'rsi_state',
    'macd_state', 'macd_hist', 'momentum_score', 'bb_pct_b', 'bollinger_state',
    'pivot_bias', 'pivot_zone', 'distance_to_pivot_pct'
]

momentum_leaders = analysis.head(10)
oversold = analysis[analysis['rsi_state'] == 'oversold'].sort_values('rsi_14')
overbought = analysis[analysis['rsi_state'] == 'overbought'].sort_values('rsi_14', ascending=False)

display(format_overview(analysis[overview_columns]))

## Momentum Leaders

Top 10 tickers by momentum score.

In [ ]:
display(format_overview(momentum_leaders[overview_columns]))

## Oversold

Tickers with RSI in oversold territory, sorted by RSI ascending.

In [ ]:
display(format_overview(oversold[overview_columns]) if not oversold.empty else oversold)

## Overbought

Tickers with RSI in overbought territory, sorted by RSI descending.

In [ ]:
display(format_overview(overbought[overview_columns]) if not overbought.empty else overbought)

## Ticker Detail

Change `SELECTED_TICKER` and rerun this cell to inspect a different stock.

In [ ]:
SELECTED_TICKER = analysis.iloc[0]['ticker']

summary = analysis.loc[analysis['ticker'] == SELECTED_TICKER].iloc[0]
stock = history_map[SELECTED_TICKER]

summary_table = pd.DataFrame(
    {
        'metric': ['Last price', 'Date', 'RSI 14', 'RSI state', 'MACD hist', 'MACD state', 'Pivot bias', 'Pivot zone', 'Currency', 'Exchange'],
        'value': [
            f"{summary['price']:,.2f}",
            summary['date'],
            f"{summary['rsi_14']:.1f}",
            summary['rsi_state'],
            f"{summary['macd_hist']:.3f}",
            summary['macd_state'],
            summary['pivot_bias'],
            summary['pivot_zone'],
            summary['currency'] or '-',
            summary['exchange'] or '-',
        ],
    }
)

pivot_table = pd.DataFrame(
    {
        'level': ['S2', 'S1', 'Pivot', 'R1', 'R2'],
        'value': [summary['pivot_s2'], summary['pivot_s1'], summary['pivot'], summary['pivot_r1'], summary['pivot_r2']],
    }
)

display(summary_table)
display(pivot_table.style.format({'value': '{:,.2f}'}, na_rep='-'))
build_price_figure(stock, summary)
